# Notebook 09 — Embedded Methods for Feature Selection

**Difficulty:** ⭐⭐⭐⭐ | **Estimated Time:** 60 minutes

---

## Learning Objectives

By the end of this notebook you will be able to:

1. Use **Lasso coefficient paths** to understand how regularization controls feature selection
2. Implement **Permutation Importance** from scratch and understand why it is unbiased
3. Compute **MDI (Mean Decrease in Impurity)** importance from scratch
4. Demonstrate **MDI bias** toward high-cardinality features and explain why permutation importance is preferred
5. Compare Lasso, MDI, Permutation, and SHAP in a unified feature-ranking table

---

**Week 12 Theme:** Airbnb-style listing price prediction

## The Analogy: The Chef Who Also Shops

Recall the two approaches to feature selection we have seen so far:

- **Filter methods** are like a nutritionist who screens ingredients before cooking — fast, but context-free.
- **Wrapper methods** are like a chef who hires trial cooks and measures dish quality — accurate, but expensive.

**Embedded methods** are like a chef who **also decides which ingredients are needed while cooking**. Selection and learning happen in the same step:

- **Lasso regression:** The L1 penalty forces a budget on the total coefficient magnitude. Like a chef constrained to spend only $50 on ingredients — she automatically skips the expensive truffles (noise features) and doubles down on the essentials.

- **Tree-based importance (MDI):** At every split, the decision tree chooses the best feature to split on. After training, we tally up how much each feature *reduced prediction error* across all splits. Features used often for large, early splits are most important.

- **Permutation Importance:** After training, shuffle one feature column at random. If model performance collapses, that feature was essential. If nothing changes, the feature was irrelevant. Unlike MDI, it is evaluated on held-out data — unbiased.

## Why Does This Matter for ML?

Embedded methods have two major advantages over filter and wrapper methods:

1. **Computational efficiency:** Feature selection happens *during* model training — no extra fitting passes (Lasso, tree importances).

2. **Model-aware selection:** The importance of each feature is evaluated *within the model* that will actually be used, so the selected features are guaranteed relevant for that specific model class.

**When to use each:**

| Method | Best for | Caveat |
|--------|----------|--------|
| Lasso | Linear models, p >> n | Assumes linearity |
| MDI | Random forests, fast ranking | Biased toward high-cardinality features |
| Permutation | Any model, reliable ranking | Slower than MDI |
| SHAP | When per-sample explanations needed | Slowest, but gold standard |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

from sklearn.linear_model import LassoCV, LinearRegression, lasso_path
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score, KFold, train_test_split
from sklearn.metrics import r2_score
from sklearn.preprocessing import StandardScaler
from scipy.stats import spearmanr

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')

np.random.seed(42)

# ── Synthetic Airbnb dataset ─────────────────────────────────────────────────
N = 600   # listings
P = 20    # total features

# 8 truly predictive features
bedrooms        = np.random.randint(1, 6, N).astype(float)        # f0
bathrooms       = np.random.randint(1, 4, N).astype(float)        # f1
accommodates    = bedrooms + np.random.randint(0, 3, N)            # f2 (correlated with f0)
distance_center = np.abs(np.random.normal(3, 2, N))               # f3 (km)
review_score    = np.random.uniform(3.5, 5.0, N)                  # f4
host_years      = np.random.randint(0, 12, N).astype(float)       # f5
neighborhood    = np.random.uniform(1, 10, N)                     # f6
luxury_score    = neighborhood / (distance_center + 0.1)           # f7 (interaction)

useful = np.column_stack([
    bedrooms, bathrooms, accommodates, distance_center,
    review_score, host_years, neighborhood, luxury_score
])  # shape (N, 8)

# 11 pure noise features + 1 random ID column (high cardinality, zero signal)
noise  = np.random.randn(N, 11)                      # f8–f18: pure Gaussian noise
random_id = np.random.choice(1000, N, replace=False).astype(float).reshape(-1, 1)  # f19: unique IDs

X = np.hstack([useful, noise, random_id])   # (N, 20)

# Ground-truth price: linear combination of useful features + noise
true_coef = np.array([30, 20, 15, -8, 25, 5, 10, 40])
y = useful @ true_coef + np.random.normal(0, 20, N) + 80

feature_names = (
    ['bedrooms', 'bathrooms', 'accommodates', 'dist_center',
     'review_score', 'host_years', 'neighborhood', 'luxury_score'] +
    [f'noise_{i}' for i in range(11)] +
    ['random_id']   # <-- this is the problematic high-cardinality column
)

# Scale features (necessary for Lasso and fair comparison)
scaler  = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Train/validation split for permutation importance
X_train, X_val, y_train, y_val = train_test_split(
    X_scaled, y, test_size=0.25, random_state=42
)

print(f"Dataset shape       : {X.shape}")
print(f"Useful features     : {feature_names[:8]}")
print(f"Noise features      : {feature_names[8:19]}")
print(f"High-cardinality col: {feature_names[19]}  ← should have zero importance")
print(f"Price range         : ${y.min():.0f} – ${y.max():.0f}")

## 1. Lasso for Feature Selection

Ordinary linear regression minimises:

$$\text{MSE} = \frac{1}{n}\|y - X\beta\|_2^2$$

Lasso adds an **L1 penalty** on the coefficient vector:

$$\text{Lasso loss} = \frac{1}{n}\|y - X\beta\|_2^2 + \alpha \|\beta\|_1$$

In plain English: the model is penalised for the *total absolute size* of its coefficients. To minimise this penalty with a fixed budget, the model is forced to drive the least useful coefficients exactly to **zero** — effectively removing those features.

- Higher α → more aggressive zeroing → sparser model
- Lower α → weaker penalty → more features survive

`LassoCV` automatically selects the best α via cross-validation.

In [ ]:
np.random.seed(42)

# ── LassoCV: auto-select α via cross-validation ───────────────────────────────
lasso_cv = LassoCV(
    cv=5,
    random_state=42,
    max_iter=10000,
    n_alphas=100
)
lasso_cv.fit(X_scaled, y)

optimal_alpha = lasso_cv.alpha_
coefs = lasso_cv.coef_
selected_mask = coefs != 0                       # non-zero = selected
selected_lasso = np.where(selected_mask)[0]
zeroed_lasso   = np.where(~selected_mask)[0]

print(f"Optimal α (LassoCV): {optimal_alpha:.4f}")
print(f"Features retained (non-zero coef): {len(selected_lasso)}")
print(f"Features zeroed out             : {len(zeroed_lasso)}")
print()
print("Selected features:")
for idx in selected_lasso:
    tag = "USEFUL" if idx < 8 else ("HIGH-CARD" if idx == 19 else "NOISE")
    print(f"  {feature_names[idx]:>20s}  coef={coefs[idx]:+.3f}  [{tag}]")

# ── Bar chart: non-zero coefficient magnitudes ────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 4))
sorted_idx = np.argsort(np.abs(coefs))[::-1]   # sort by magnitude
bar_colors = [
    'steelblue' if i < 8 else ('firebrick' if i == 19 else 'lightgray')
    for i in sorted_idx
]
ax.bar(range(P), np.abs(coefs[sorted_idx]), color=bar_colors, edgecolor='white')
ax.set_xticks(range(P))
ax.set_xticklabels([feature_names[i] for i in sorted_idx], rotation=45, ha='right', fontsize=8)
ax.set_ylabel('|Lasso Coefficient|')
ax.set_title(f'Lasso Coefficients at Optimal α={optimal_alpha:.4f} (blue=useful, red=ID, gray=noise)')
plt.tight_layout()
plt.show()

useful_recovered = sum(1 for i in selected_lasso if i < 8)
print(f"\nTruly useful features recovered: {useful_recovered}/8")
print(f"Noise features incorrectly kept: {sum(1 for i in selected_lasso if 8 <= i < 19)}")

In [ ]:
np.random.seed(42)

# ── Lasso regularization path: how do coefficients change as α increases? ─────
# lasso_path returns: alphas, coefs (P x len(alphas))
alphas_path, coefs_path, _ = lasso_path(
    X_scaled, y,
    alphas=np.logspace(-3, 2, 100),   # α from 0.001 to 100
    max_iter=10000
)

fig, ax = plt.subplots(figsize=(11, 5))

# Plot useful features with solid lines, noise with dashed/faint lines
for i in range(P):
    if i < 8:   # truly useful
        ax.plot(np.log10(alphas_path), coefs_path[i],
                linewidth=2, label=feature_names[i])
    elif i == 19:  # random ID
        ax.plot(np.log10(alphas_path), coefs_path[i],
                linewidth=1.5, color='red', linestyle=':', label='random_id')
    else:  # noise
        ax.plot(np.log10(alphas_path), coefs_path[i],
                linewidth=0.7, color='gray', alpha=0.4)

ax.axvline(np.log10(optimal_alpha), color='black', linestyle='--',
           linewidth=1.5, label=f'Optimal α (CV)')
ax.axhline(0, color='black', linewidth=0.5)

ax.set_xlabel('log₁₀(α)  →  more regularization →')
ax.set_ylabel('Coefficient value')
ax.set_title('Lasso Regularization Path: Coefficients vs. Penalty Strength')
ax.legend(loc='upper right', fontsize=7, ncol=2)
plt.tight_layout()
plt.show()

print("Interpretation:")
print("  - Moving right = stronger penalty = more coefficients zeroed")
print("  - Features that survive to high α are the most robustly important")
print("  - Noise features (gray) are zeroed first at low α")

In [ ]:
np.random.seed(42)

# ── Compare Lasso selection vs. full model vs. unregularized ──────────────────
cv5 = KFold(n_splits=5, shuffle=True, random_state=42)

# Model 1: All features, unregularized linear regression
lr_all = LinearRegression()
r2_all = cross_val_score(lr_all, X_scaled, y, cv=cv5, scoring='r2').mean()

# Model 2: Lasso-selected features, then refit with plain LinearRegression
lr_lasso = LinearRegression()
r2_lasso = cross_val_score(lr_lasso, X_scaled[:, selected_lasso], y,
                           cv=cv5, scoring='r2').mean()

# Model 3: Oracle — only the 8 truly useful features
lr_oracle = LinearRegression()
r2_oracle = cross_val_score(lr_oracle, X_scaled[:, :8], y,
                            cv=cv5, scoring='r2').mean()

# Model 4: Lasso model itself (with L1 penalty, at optimal α)
r2_lasso_model = cross_val_score(lasso_cv, X_scaled, y, cv=cv5, scoring='r2').mean()

print("5-Fold CV R² comparison:")
print(f"  All 20 features (OLS)          : {r2_all:.4f}")
print(f"  Lasso-selected features (OLS)  : {r2_lasso:.4f}")
print(f"  Lasso model (with L1)          : {r2_lasso_model:.4f}")
print(f"  Oracle (8 true features, OLS)  : {r2_oracle:.4f}")
print()
print("Lasso achieves near-oracle performance by zeroing out noise features.")
print("OLS on all 20 features overfits slightly to the 12 noise columns.")

## 2. Tree-Based MDI Importance (Mean Decrease in Impurity)

When a decision tree splits on feature j at node v, it reduces impurity (MSE for regression) by:

$$\Delta I_v = \frac{n_v}{n} \cdot \text{MSE}(v) - \frac{n_L}{n} \cdot \text{MSE}(L) - \frac{n_R}{n} \cdot \text{MSE}(R)$$

In plain English: the weighted impurity of the parent minus the weighted impurity of the two children.

**MDI importance** for feature j = sum of these reductions across all nodes that split on j, averaged over all trees in the forest.

**Why MDI is fast:** These numbers are computed as a byproduct of training — no extra model fits needed.

In [ ]:
np.random.seed(42)

# ── Train a Random Forest on the Airbnb data ─────────────────────────────────
rf_model = RandomForestRegressor(
    n_estimators=200,
    max_depth=8,
    random_state=42,
    n_jobs=-1
)
rf_model.fit(X_train, y_train)   # fit on training split

print(f"RF validation R²: {r2_score(y_val, rf_model.predict(X_val)):.4f}")

# ── MDI from scratch ──────────────────────────────────────────────────────────
def mdi_importance_scratch(forest, n_features):
    """Compute Mean Decrease in Impurity importance from a RandomForest.
    
    For each internal node in every tree:
      impurity_decrease = (n_node/N)*MSE(node) - (n_left/N)*MSE(left) - (n_right/N)*MSE(right)
    Accumulate by feature, average over trees, normalize to sum=1.
    """
    importances = np.zeros(n_features)          # accumulate per-feature importance

    for tree in forest.estimators_:             # iterate over all trees in the forest
        tree_ = tree.tree_                      # access the underlying Tree object

        for node in range(tree_.node_count):
            # Leaf nodes have children_left == -1; skip them
            if tree_.children_left[node] == -1:
                continue

            feat   = tree_.feature[node]                                  # feature used at this split
            n_node = tree_.n_node_samples[node]                           # samples reaching this node
            n_left = tree_.n_node_samples[tree_.children_left[node]]      # samples going left
            n_right= tree_.n_node_samples[tree_.children_right[node]]     # samples going right
            n_root = tree_.n_node_samples[0]                              # total samples in root

            # Weighted impurity decrease (normalised by root sample count)
            delta = (
                n_node  * tree_.impurity[node] -
                n_left  * tree_.impurity[tree_.children_left[node]] -
                n_right * tree_.impurity[tree_.children_right[node]]
            ) / n_root

            importances[feat] += delta          # add to the responsible feature

    importances /= len(forest.estimators_)      # average over trees
    importances /= (importances.sum() + 1e-10)  # normalize to sum=1
    return importances


mdi_scratch = mdi_importance_scratch(rf_model, n_features=P)
mdi_sklearn = rf_model.feature_importances_    # sklearn's built-in MDI

# Check agreement
max_diff = np.max(np.abs(mdi_scratch - mdi_sklearn))
print(f"\nMax absolute diff between scratch and sklearn MDI: {max_diff:.2e}")
print("(Should be near zero — same algorithm, same tree structures)")

## MDI Bias: Why High-Cardinality Features Get Inflated Importance

MDI has a well-known bias: **features with many unique values get more splits, so they accumulate more impurity reduction — even if they carry no real signal.**

Our `random_id` feature has 600 unique values in a dataset of 600 rows. A tree can always split on it to perfectly separate any pair of samples. During training, the tree greedily uses it because it appears to reduce MSE. But this is pure overfitting to the training data.

**The fix:** Evaluate importance on **held-out data** (permutation importance).

In [ ]:
np.random.seed(42)

# ── Demonstrate MDI bias on the random_id column ─────────────────────────────
id_importance_mdi = mdi_sklearn[19]   # random_id is index 19

fig, ax = plt.subplots(figsize=(11, 4))

sorted_idx = np.argsort(mdi_sklearn)[::-1]
bar_colors = [
    'steelblue' if i < 8 else ('firebrick' if i == 19 else 'lightgray')
    for i in sorted_idx
]
ax.bar(range(P), mdi_sklearn[sorted_idx], color=bar_colors, edgecolor='white')
ax.set_xticks(range(P))
ax.set_xticklabels([feature_names[i] for i in sorted_idx], rotation=45, ha='right', fontsize=8)
ax.set_ylabel('MDI Importance (normalized)')
ax.set_title('MDI Importance — random_id (red) gets non-zero importance despite carrying no signal!')

# Highlight the ID column with an annotation
id_position = list(sorted_idx).index(19)
ax.annotate(
    f'random_id\n(zero signal!\nMDI={id_importance_mdi:.4f})',
    xy=(id_position, id_importance_mdi),
    xytext=(id_position + 1.5, id_importance_mdi + 0.02),
    fontsize=8, color='firebrick',
    arrowprops=dict(arrowstyle='->', color='firebrick')
)

plt.tight_layout()
plt.show()

print(f"random_id MDI importance: {id_importance_mdi:.6f}")
print(f"Average noise feature MDI: {mdi_sklearn[8:19].mean():.6f}")
print(f"random_id vs. avg noise: {id_importance_mdi / mdi_sklearn[8:19].mean():.1f}x higher")
print("\nThis is MDI bias — high cardinality inflates apparent importance.")

## 3. Permutation Importance (Unbiased)

**Algorithm:**
1. Fit the model and record **baseline performance** on the validation set
2. For each feature j:
   a. **Shuffle** column j randomly (breaking its relationship with y)
   b. Record the **performance drop** (baseline − shuffled performance)
   c. Repeat `n_repeats` times and average
3. Feature importance = mean performance drop when that feature is shuffled

**Why this is unbiased:**
- Evaluated on **held-out validation data** → no overfitting inflation
- High-cardinality features that were memorised on training data show **zero importance** on validation data when shuffled
- Works with **any model** — it only requires `model.predict()`

**The only caveat:** Correlated features can share importance. Shuffling feature A might not hurt much if feature B (correlated with A) compensates.

In [ ]:
np.random.seed(42)

def permutation_importance_scratch(model, X_val, y_val, n_repeats=10):
    """Compute permutation importance on held-out validation data.
    
    Parameters
    ----------
    model     : fitted model with .predict() method
    X_val     : ndarray (n_val, n_features) — HELD-OUT data
    y_val     : ndarray (n_val,)
    n_repeats : int, number of shuffle repetitions per feature
    
    Returns
    -------
    mean_imp : ndarray (n_features,), mean importance (performance drop)
    std_imp  : ndarray (n_features,), standard deviation across repeats
    """
    # Baseline R² on unmodified validation set
    baseline = r2_score(y_val, model.predict(X_val))

    importances = np.zeros((X_val.shape[1], n_repeats))  # (n_features, n_repeats)

    for j in range(X_val.shape[1]):           # iterate over each feature
        for r in range(n_repeats):            # repeat the shuffle n_repeats times
            X_perm = X_val.copy()             # fresh copy each time
            X_perm[:, j] = np.random.permutation(X_perm[:, j])   # shuffle column j only
            perm_score = r2_score(y_val, model.predict(X_perm))
            importances[j, r] = baseline - perm_score             # drop = importance

    return importances.mean(axis=1), importances.std(axis=1)


perm_mean, perm_std = permutation_importance_scratch(
    rf_model, X_val, y_val, n_repeats=20
)

print(f"Baseline validation R²: {r2_score(y_val, rf_model.predict(X_val)):.4f}")
print(f"random_id permutation importance: {perm_mean[19]:.6f} ± {perm_std[19]:.6f}")
print(f"(Expected ≈ 0 — shuffling a zero-signal feature should not hurt performance)")

# ── Horizontal bar chart: importance ± std ────────────────────────────────────
sorted_idx_perm = np.argsort(perm_mean)         # ascending order for horizontal bar
bar_colors_perm = [
    'steelblue' if i < 8 else ('firebrick' if i == 19 else 'lightgray')
    for i in sorted_idx_perm
]

fig, ax = plt.subplots(figsize=(8, 7))
y_pos = range(P)
ax.barh(
    y_pos, perm_mean[sorted_idx_perm],
    xerr=perm_std[sorted_idx_perm],
    color=bar_colors_perm, edgecolor='white',
    capsize=3, height=0.65
)
ax.set_yticks(y_pos)
ax.set_yticklabels([feature_names[i] for i in sorted_idx_perm], fontsize=8)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Mean R² drop when feature is shuffled (importance ± std)')
ax.set_title('Permutation Importance on Held-Out Data\n(blue=useful, red=random_id, gray=noise)')
plt.tight_layout()
plt.show()

In [ ]:
np.random.seed(42)

# ── MDI vs. Permutation: side-by-side comparison ─────────────────────────────
# Key question: does random_id get near-zero permutation importance but non-zero MDI?

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Shared feature ordering by permutation importance
order = np.argsort(perm_mean)   # ascending, for horizontal bar
colors = ['steelblue' if i < 8 else ('firebrick' if i == 19 else 'lightgray') for i in order]
y_pos = range(P)

# Panel 1: MDI importance
ax1.barh(y_pos, mdi_sklearn[order], color=colors, edgecolor='white', height=0.65)
ax1.set_yticks(y_pos)
ax1.set_yticklabels([feature_names[i] for i in order], fontsize=8)
ax1.set_xlabel('MDI importance')
ax1.set_title('MDI Importance\n(biased toward random_id)')
ax1.axvline(0, color='black', linewidth=0.5)

# Panel 2: Permutation importance
ax2.barh(y_pos, perm_mean[order], xerr=perm_std[order],
         color=colors, edgecolor='white', capsize=3, height=0.65)
ax2.set_yticks(y_pos)
ax2.set_yticklabels([feature_names[i] for i in order], fontsize=8)
ax2.set_xlabel('Permutation importance (R² drop)')
ax2.set_title('Permutation Importance\n(random_id correctly ≈ 0)')
ax2.axvline(0, color='black', linewidth=0.5)

plt.suptitle('MDI vs. Permutation Importance:\nMDI Inflates High-Cardinality Features', fontweight='bold')
plt.tight_layout()
plt.show()

# Spearman rank correlation between MDI and Permutation
rho, pval = spearmanr(mdi_sklearn, perm_mean)
print(f"Spearman ρ (MDI vs. Permutation): {rho:.3f}  (p={pval:.4f})")
print()
print(f"random_id MDI importance  : {mdi_sklearn[19]:.6f}")
print(f"random_id Perm importance : {perm_mean[19]:.6f} ± {perm_std[19]:.6f}")
print()
print("Conclusion: use permutation importance whenever cardinality varies across features.")

## 4. SHAP Values (Preview)

SHAP (SHapley Additive exPlanations) is the **gold standard** for feature importance.

**Intuition:** In cooperative game theory, Shapley values assign a fair credit to each player for the team's total output. SHAP applies this idea to ML: each feature's Shapley value = its average marginal contribution across all possible feature coalitions.

**Advantages over MDI and Permutation:**
- **Per-sample explanations:** Each prediction has its own SHAP vector
- **Theoretically grounded:** Uniquely satisfies four desirable axioms (efficiency, symmetry, dummy, linearity)
- **Unbiased:** Evaluated via model evaluation, not training statistics
- **Detects interactions:** SHAP interaction values available

**Cost:** O(2^p) in general; O(n × p × depth) for trees (via TreeSHAP — fast).

In [ ]:
np.random.seed(42)

# ── SHAP: use if installed, else fall back to permutation importance ──────────
try:
    import shap

    # TreeExplainer is fast for Random Forests — uses TreeSHAP algorithm
    explainer = shap.TreeExplainer(rf_model)
    shap_values = explainer.shap_values(X_val)   # shape: (n_val, n_features)

    # Summary plot: beeswarm of SHAP values — shows direction AND magnitude
    fig, ax = plt.subplots(figsize=(8, 6))
    shap.summary_plot(
        shap_values, X_val,
        feature_names=feature_names,
        show=False, plot_size=None
    )
    plt.title('SHAP Summary Plot (TreeSHAP)')
    plt.tight_layout()
    plt.show()

    # Mean absolute SHAP = global feature importance
    shap_importance = np.abs(shap_values).mean(axis=0)
    print("SHAP installed — TreeSHAP computed successfully.")
    print(f"random_id SHAP importance: {shap_importance[19]:.6f}  (expected ≈ 0)")

except ImportError:
    # Graceful fallback: show permutation importance as proxy
    print("shap not installed.")
    print("Showing permutation importance as a reliable proxy.")
    print()
    print("To install: pip install shap")
    print()
    print("Permutation importance (already computed):")
    sorted_by_perm = np.argsort(perm_mean)[::-1]
    for rank, idx in enumerate(sorted_by_perm[:10], 1):
        tag = "USEFUL" if idx < 8 else ("HIGH-CARD" if idx == 19 else "NOISE")
        print(f"  Rank {rank:2d}: {feature_names[idx]:>20s}  importance={perm_mean[idx]:.4f}  [{tag}]")
    shap_importance = None   # not available

In [ ]:
np.random.seed(42)

# ── Method comparison table ───────────────────────────────────────────────────
# Rank each feature by Lasso |coef|, MDI, Permutation, and SHAP (if available)

# Ranks (1 = most important)
lasso_rank = pd.Series(np.abs(lasso_cv.coef_), index=feature_names).rank(ascending=False).astype(int)
mdi_rank   = pd.Series(mdi_sklearn,             index=feature_names).rank(ascending=False).astype(int)
perm_rank  = pd.Series(perm_mean,               index=feature_names).rank(ascending=False).astype(int)

comparison = pd.DataFrame({
    'Lasso |coef|': np.abs(lasso_cv.coef_),
    'Lasso Rank':   lasso_rank,
    'MDI':          mdi_sklearn,
    'MDI Rank':     mdi_rank,
    'Perm Imp':     perm_mean,
    'Perm Rank':    perm_rank,
}, index=feature_names)

# Add SHAP if available
if shap_importance is not None:
    shap_rank = pd.Series(shap_importance, index=feature_names).rank(ascending=False).astype(int)
    comparison['SHAP']      = shap_importance
    comparison['SHAP Rank'] = shap_rank

# Sort by permutation rank (most reliable ground truth)
comparison = comparison.sort_values('Perm Rank')

print("Feature Importance Comparison (sorted by Permutation rank):")
print(comparison.to_string(float_format=lambda x: f'{x:.4f}'))

# Pairwise Spearman correlations
print("\nSpearman rank correlations:")
rho_lasso_mdi,  _ = spearmanr(np.abs(lasso_cv.coef_), mdi_sklearn)
rho_lasso_perm, _ = spearmanr(np.abs(lasso_cv.coef_), perm_mean)
rho_mdi_perm,   _ = spearmanr(mdi_sklearn, perm_mean)

print(f"  Lasso vs. MDI         : ρ = {rho_lasso_mdi:.3f}")
print(f"  Lasso vs. Permutation : ρ = {rho_lasso_perm:.3f}")
print(f"  MDI   vs. Permutation : ρ = {rho_mdi_perm:.3f}")

In [ ]:
np.random.seed(42)

# ── Embedded selection pipeline ──────────────────────────────────────────────
# Step 1: Lasso pre-selects features (fast, embedded)
# Step 2: Among Lasso-selected features, rank by Permutation Importance
# Step 3: Retain top-k
# Compare CV R² vs. using all features

cv5 = KFold(n_splits=5, shuffle=True, random_state=42)

# ── Step 1: Lasso pre-selection ──
lasso_selected_idx = np.where(lasso_cv.coef_ != 0)[0]
print(f"Step 1 — Lasso pre-selected: {len(lasso_selected_idx)} features")
print(f"  {[feature_names[i] for i in lasso_selected_idx]}")

# ── Step 2: Permutation importance within Lasso-selected features ──
# Refit RF on train using only Lasso-selected features
rf_lasso = RandomForestRegressor(n_estimators=200, max_depth=8, random_state=42, n_jobs=-1)
rf_lasso.fit(X_train[:, lasso_selected_idx], y_train)

perm_mean_sub, perm_std_sub = permutation_importance_scratch(
    rf_lasso, X_val[:, lasso_selected_idx], y_val, n_repeats=15
)

# ── Step 3: Retain top-8 by permutation importance ──
top_k = min(8, len(lasso_selected_idx))
top_local_idx = np.argsort(perm_mean_sub)[::-1][:top_k]   # local indices within lasso_selected_idx
final_features = lasso_selected_idx[top_local_idx]         # map back to original indices

print(f"\nStep 2+3 — Top-{top_k} by permutation importance within Lasso set:")
for rank, idx in enumerate(final_features, 1):
    tag = "USEFUL" if idx < 8 else ("HIGH-CARD" if idx == 19 else "NOISE")
    print(f"  Rank {rank}: {feature_names[idx]:>20s}  [{tag}]")

# ── CV R² comparison ──
rf_baseline = RandomForestRegressor(n_estimators=200, max_depth=8, random_state=42, n_jobs=-1)
rf_pipeline = RandomForestRegressor(n_estimators=200, max_depth=8, random_state=42, n_jobs=-1)
rf_oracle   = RandomForestRegressor(n_estimators=200, max_depth=8, random_state=42, n_jobs=-1)

r2_baseline = cross_val_score(rf_baseline, X_scaled,                 y, cv=cv5, scoring='r2').mean()
r2_pipeline = cross_val_score(rf_pipeline, X_scaled[:, final_features], y, cv=cv5, scoring='r2').mean()
r2_oracle2  = cross_val_score(rf_oracle,   X_scaled[:, :8],           y, cv=cv5, scoring='r2').mean()

print(f"\nCV R² — All 20 features   : {r2_baseline:.4f}")
print(f"CV R² — Pipeline (top-{top_k})  : {r2_pipeline:.4f}")
print(f"CV R² — Oracle (true 8)   : {r2_oracle2:.4f}")
print()
print("The embedded pipeline achieves competitive R² with fewer features and less noise.")

## Self-Check Questions

---

**Q1.** You fit a Random Forest and find that `customer_id` (a unique integer per row) has the **highest MDI importance** but near-zero **permutation importance** on the validation set. What is happening and what should you do?

<details><summary>Show answer</summary>

This is **MDI bias in action**. `customer_id` is a high-cardinality feature — it has as many unique values as there are training rows. The tree can split on it to almost perfectly separate training samples, so it accumulates enormous impurity reduction on the training set. But because these splits memorise training IDs (not a real pattern), the feature provides zero predictive power on held-out data.

**What to do:**
1. **Drop the ID column before training** — this is the root fix. IDs should never be features.
2. **Always use permutation importance** (or SHAP) for feature ranking instead of MDI.
3. If you must keep high-cardinality categoricals, encode them carefully (target encoding with CV, not raw integer encoding).

</details>

---

**Q2.** Lasso zeroed out `feature_A`, but permutation importance shows `feature_A` has large positive importance. Is this a contradiction? What could explain it?

<details><summary>Show answer</summary>

Not a contradiction — these measure different things:

- **Lasso** measures linear importance within a **linear model** using **regularized regression**. If `feature_A` is highly correlated with another feature that Lasso kept, Lasso will zero out the redundant one arbitrarily.

- **Permutation importance** is computed on a **Random Forest** which can capture non-linear effects.

**Possible explanations:**
1. `feature_A` has a **non-linear** relationship with y that Lasso (a linear model) cannot capture but the Random Forest can
2. `feature_A` is **correlated** with a feature that Lasso kept — both carry the same linear signal, so Lasso drops one. The RF uses both.
3. `feature_A` is involved in a **feature interaction** that only the RF detects

**Lesson:** Always evaluate importance relative to the model you will actually deploy.

</details>

---

**Q3.** Two features A and B are perfectly correlated (ρ=1). You shuffle feature A for permutation importance. Why might the permutation importance of A be near zero even if both A and B are genuinely predictive?

<details><summary>Show answer</summary>

When A and B are perfectly correlated, shuffling A does **not remove the signal** — the model can still use B to recover the same prediction. So the performance drop is near zero, giving A near-zero permutation importance.

This is the **correlated features problem** with permutation importance:
- The total importance is split between A and B
- Individually, neither looks important, but removing **both** would cause a large performance drop

**Solutions:**
1. Remove one of the correlated features before computing importance
2. Use **conditional permutation importance** (shuffle A holding B constant)
3. Use **SHAP interaction values** which properly allocate credit between correlated features
4. Use **hierarchical clustering on correlation matrix** to group correlated features, then select one representative per group

</details>

## Key Takeaways

1. **Lasso is the cleanest embedded method for linear models.** It automatically zeros out irrelevant features during training and the regularization path shows you exactly which features survive at each penalty level.

2. **MDI is fast but biased.** High-cardinality features (IDs, zip codes, timestamps as integers) get artificially inflated MDI scores because they accumulate more splits. Never use MDI as your final importance ranking.

3. **Permutation importance is unbiased** because it is computed on held-out data. If a feature was memorised on training data (overfitting), shuffling it on validation data shows zero importance.

4. **SHAP is the gold standard** when interpretability matters — it provides per-sample explanations, is theoretically grounded, and properly handles feature interactions. The cost is acceptable for tree models via TreeSHAP.

5. **Practical embedded pipeline:**
   - For linear models: use LassoCV → refit LinearRegression on non-zero features
   - For tree models: fit RandomForest → compute permutation importance → select top-k → refit
   - When you need explanations: add SHAP on top of any of the above

6. **All methods agree on the top features** (high Spearman ρ at the top) but disagree on borderline and noisy features. When methods disagree, permutation importance and SHAP are the more reliable arbiters.

---

**Up next:** Notebook 10 — Building the Full ML Pipeline (putting it all together)